# Clonagem e validacao quantitativa de subcomunidades

Use este notebook para passar um caminho `.pt`, clonar o grafo com `GenCAT`, `SkyMap`, `Chung-Lu` e `SynCo`, e avaliar quantitativamente se os clones preservam a estrutura interna das comunidades.

Fluxo do notebook:

1. voce configura `BASE_GRAPH_PATH`;
2. o grafo original e carregado como `AttributedGraph`;
3. cada modelo em `models/` clona o grafo;
4. os clones sao salvos em disco;
5. as subcomunidades sao detectadas dentro de cada classe;
6. as metricas sao calculadas contra o grafo original;
7. tabelas e figuras sao salvas em `experiments/results/subcommunity_validation_runs/`.

In [ ]:
from pathlib import Path
from datetime import datetime

ROOT = Path.cwd()
if not (ROOT / "models").exists():
    ROOT = ROOT.parent

# Altere somente este caminho para avaliar outro grafo .pt.
BASE_GRAPH_PATH = ROOT / "datasets" / "Cora" / "processed" / "data.pt"

# Modelos disponiveis: "SynCo", "GenCAT", "Chung-Lu", "SkyMap".
# Se algum modelo estiver muito demorado, remova temporariamente da lista.
MODELS_TO_RUN = ["SynCo", "GenCAT", "Chung-Lu", "SkyMap"]

SEED = 2026
SAVE_CLONES = True
STOP_ON_MODEL_ERROR = False
LOUVAIN_RESOLUTION = 1.0

RUN_ID = f"{BASE_GRAPH_PATH.parent.parent.name}_{BASE_GRAPH_PATH.stem}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUTPUT_DIR = ROOT / "experiments" / "results" / "subcommunity_validation_runs" / RUN_ID
CLONES_DIR = OUTPUT_DIR / "clones"
FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR = OUTPUT_DIR / "tables"

for directory in [OUTPUT_DIR, CLONES_DIR, FIGURES_DIR, TABLES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Base graph: {BASE_GRAPH_PATH}")
print(f"Output dir: {OUTPUT_DIR}")

## O que cada metrica mede

Nas metricas de erro ou distancia, menor e melhor. Em `nmi` e `ari`, maior e melhor.

- `modularity_abs_error`: erro absoluto entre a modularidade das subcomunidades do original e do clone. Mede se a separacao interna das subcomunidades ficou tao forte quanto no grafo original.
- `degree_wasserstein`: distancia entre distribuicoes de graus dos nos dentro das subcomunidades alinhadas. Mede preservacao do perfil de conectividade local.
- `clustering_wasserstein`: distancia entre distribuicoes de coeficiente de agrupamento local. Mede preservacao de fechamento triadico.
- `triangle_wasserstein`: distancia entre distribuicoes de triangulos por no. Mede preservacao de motivos locais simples.
- `interaction_frobenius`: distancia de Frobenius entre matrizes normalizadas de interacao entre subcomunidades. Mede se o padrao de conexao entre subcomunidades foi preservado.
- `interaction_l1`: distancia L1 entre as mesmas matrizes de interacao. Mede a diferenca total de massa entre os padroes de interacao.
- `subcommunity_size_abs_error`: erro absoluto medio no tamanho das subcomunidades alinhadas. Mede preservacao da escala das subcomunidades.
- `nmi`: Normalized Mutual Information entre as particoes detectadas no original e no clone, quando os IDs dos nos sao comparaveis. Mede similaridade entre particionamentos.
- `ari`: Adjusted Rand Index entre as particoes detectadas, ajustado pelo acaso. Mede similaridade entre particionamentos com penalizacao para coincidencias aleatorias.

In [ ]:
import sys
import math
import random
import time
import warnings

import networkit as nk
import networkx as nx
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from scipy.optimize import linear_sum_assignment
from scipy.stats import wasserstein_distance
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from torch_geometric.data import Data
from torch_geometric.utils import from_networkit, to_networkit

try:
    import seaborn as sns
    HAS_SEABORN = True
except Exception:
    HAS_SEABORN = False

sys.path.insert(0, str(ROOT))

from synco import AttributedGraph
from models.GenCAT import GenCATGenerator
from synco import SynCoGenerator
from models.SkyMap import SkyMapGenerator
from models.chunglu import ChungLuGenerator

warnings.filterwarnings("ignore", category=RuntimeWarning)

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 300,
    "font.size": 10,
})

## Funcoes auxiliares

Estas funcoes carregam o `.pt`, padronizam o objeto para `AttributedGraph`, convertem para `networkx` para analise, e controlam sementes para reproducibilidade.

In [ ]:
def set_all_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    try:
        nk.setSeed(seed, useThreadId=False)
    except Exception:
        pass


def torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def get_data_field(data, name):
    value = getattr(data, name, None)
    if value is not None:
        return value

    try:
        if name in list(data.keys()):
            return data[name]
    except Exception:
        pass

    raise KeyError(f"Campo ausente no .pt: {name}")


def unpack_pt_object(obj):
    if isinstance(obj, AttributedGraph):
        return obj.x.detach().cpu(), obj.y.detach().cpu().view(-1).long(), from_networkit(obj.graph)[0].detach().cpu().long()

    data = obj[0] if isinstance(obj, tuple) else obj
    x = get_data_field(data, "x").detach().cpu()
    y = get_data_field(data, "y").detach().cpu().view(-1).long()
    edge_index = get_data_field(data, "edge_index").detach().cpu().long()
    return x, y, edge_index


def edge_index_to_nx(edge_index, num_nodes):
    graph = nx.Graph()
    graph.add_nodes_from(range(num_nodes))

    for u, v in edge_index.t().cpu().numpy():
        u = int(u)
        v = int(v)
        if u != v:
            graph.add_edge(u, v)

    return graph


def attributed_graph_from_pt(path):
    x, y, edge_index = unpack_pt_object(torch_load(path))
    graph = to_networkit(edge_index, directed=False)
    return AttributedGraph(graph=graph, x=x, y=y)


def record_from_attributed_graph(name, attributed_graph, path=None):
    data = attributed_graph.to_data_pytorch()
    x = data.x.detach().cpu()
    y = data.y.detach().cpu().view(-1).long()
    edge_index = data.edge_index.detach().cpu().long()
    graph = edge_index_to_nx(edge_index, int(y.numel()))

    return {
        "name": name,
        "path": str(path) if path is not None else "in_memory",
        "x": x,
        "y": y.numpy(),
        "edge_index": edge_index,
        "graph": graph,
        "num_nodes": graph.number_of_nodes(),
        "num_edges": graph.number_of_edges(),
    }

## Carregar grafo original

A celula abaixo carrega o `.pt` definido em `BASE_GRAPH_PATH` e cria o registro que sera usado como referencia para todos os clones.

In [ ]:
set_all_seeds(SEED)

base_graph = attributed_graph_from_pt(BASE_GRAPH_PATH)
original_record = record_from_attributed_graph("original", base_graph, path=BASE_GRAPH_PATH)

print(base_graph)
print(f"Classes: {sorted(np.unique(original_record['y']).tolist())}")
print(f"Arestas nao direcionadas usadas na analise: {original_record['num_edges']}")

## Clonar com os modelos em `models/`

Esta celula chama diretamente os quatro geradores. Se um modelo falhar e `STOP_ON_MODEL_ERROR=False`, o notebook registra o erro e continua para os proximos modelos.

In [ ]:
def clone_with_model(model_name, graph_to_clone):
    if model_name == "SynCo":
        return SynCoGenerator(seed=SEED).mimic(graph_to_clone)

    if model_name == "GenCAT":
        return GenCATGenerator().mimic(
            graph_to_clone,
            feature_dim=int(graph_to_clone.x.shape[1]),
        )

    if model_name == "Chung-Lu":
        return ChungLuGenerator(seed=SEED).mimic(
            graph_to_clone,
            seed=SEED,
            copy_x=True,
        )

    if model_name == "SkyMap":
        return SkyMapGenerator().mimic(graph_to_clone)

    raise ValueError(f"Modelo desconhecido: {model_name}")


clone_records = {}
clone_log_rows = []

for model_name in MODELS_TO_RUN:
    print(f"Clonando com {model_name}...", flush=True)
    set_all_seeds(SEED)
    start = time.time()

    try:
        cloned_graph = clone_with_model(model_name, base_graph)
        elapsed = time.time() - start

        clone_path = CLONES_DIR / f"{model_name}.pt"
        if SAVE_CLONES:
            torch.save(cloned_graph.to_data_pytorch(), clone_path)

        clone_record = record_from_attributed_graph(
            model_name,
            cloned_graph,
            path=clone_path if SAVE_CLONES else None,
        )
        clone_records[model_name] = clone_record

        clone_log_rows.append({
            "model": model_name,
            "status": "ok",
            "seconds": elapsed,
            "nodes": clone_record["num_nodes"],
            "edges": clone_record["num_edges"],
            "path": str(clone_path) if SAVE_CLONES else "not_saved",
            "error": "",
        })

        print(f"  OK: {clone_record['num_nodes']} nos, {clone_record['num_edges']} arestas, {elapsed:.1f}s", flush=True)

    except Exception as exc:
        elapsed = time.time() - start
        clone_log_rows.append({
            "model": model_name,
            "status": "error",
            "seconds": elapsed,
            "nodes": np.nan,
            "edges": np.nan,
            "path": "",
            "error": repr(exc),
        })
        print(f"  ERRO em {model_name}: {exc!r}", flush=True)
        if STOP_ON_MODEL_ERROR:
            raise

clone_log = pd.DataFrame(clone_log_rows)
clone_log.to_csv(TABLES_DIR / "clone_log.csv", index=False)
clone_log

## Deteccao de subcomunidades

Para cada grafo, a analise separa os nos por classe (`y`) e detecta subcomunidades dentro do subgrafo induzido por cada classe. A deteccao usa Louvain em `networkx`.

In [ ]:
def detect_communities(subgraph, seed=SEED, resolution=LOUVAIN_RESOLUTION):
    if subgraph.number_of_nodes() == 0:
        return []
    if subgraph.number_of_edges() == 0:
        return [{node} for node in subgraph.nodes()]

    try:
        communities = nx.community.louvain_communities(
            subgraph,
            seed=seed,
            resolution=resolution,
        )
    except Exception:
        communities = nx.community.greedy_modularity_communities(subgraph)

    communities = [set(community) for community in communities if len(community) > 0]
    communities.sort(key=lambda community: (-len(community), min(community)))
    return communities


def partition_vector(num_nodes, communities):
    labels = np.full(num_nodes, -1, dtype=int)
    for cid, community in enumerate(communities):
        labels[list(community)] = cid
    return labels


def detect_subcommunities(record):
    graph = record["graph"]
    y = record["y"]
    class_labels = sorted(int(label) for label in np.unique(y) if int(label) >= 0)

    by_class = {}
    for label in class_labels:
        nodes = np.flatnonzero(y == label).tolist()
        class_graph = graph.subgraph(nodes).copy()
        communities = detect_communities(class_graph)
        by_class[label] = {
            "nodes": nodes,
            "graph": class_graph,
            "communities": communities,
            "partition": partition_vector(record["num_nodes"], communities),
        }

    record["subcommunities"] = by_class
    return record


all_records = {"original": original_record, **clone_records}
for record in all_records.values():
    detect_subcommunities(record)

subcommunity_counts = pd.DataFrame([
    {
        "model": name,
        "class": label,
        "class_nodes": len(info["nodes"]),
        "num_subcommunities": len(info["communities"]),
    }
    for name, record in all_records.items()
    for label, info in record["subcommunities"].items()
])

subcommunity_counts.to_csv(TABLES_DIR / "subcommunity_counts.csv", index=False)
subcommunity_counts

## Calculo das metricas

As funcoes abaixo resumem cada subcomunidade, alinham subcomunidades do clone com as do original, e calculam as distancias classe a classe.

In [ ]:
def safe_mean(values):
    values = np.asarray(values, dtype=float)
    return float("nan") if values.size == 0 else float(np.mean(values))


def safe_std(values):
    values = np.asarray(values, dtype=float)
    return float("nan") if values.size == 0 else float(np.std(values))


def safe_density(num_nodes, num_edges):
    denom = num_nodes * (num_nodes - 1) / 2
    return float(num_edges / denom) if denom > 0 else float("nan")


def safe_modularity(graph, communities):
    if graph.number_of_edges() == 0 or len(communities) <= 1:
        return float("nan")
    try:
        return float(nx.community.modularity(graph, communities))
    except Exception:
        return float("nan")


def safe_wasserstein(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if a.size == 0 or b.size == 0:
        return float("nan")
    return float(wasserstein_distance(a, b))


def interaction_matrix(graph, communities):
    k = len(communities)
    matrix = np.zeros((k, k), dtype=float)
    node_to_community = {}

    for cid, community in enumerate(communities):
        for node in community:
            node_to_community[node] = cid

    for u, v in graph.edges():
        cu = node_to_community.get(u)
        cv = node_to_community.get(v)
        if cu is None or cv is None:
            continue
        matrix[cu, cv] += 1
        if cu != cv:
            matrix[cv, cu] += 1
        else:
            matrix[cu, cv] += 1

    total = matrix.sum()
    if total > 0:
        matrix = matrix / total
    return matrix


def padded_matrix_distance(a, b):
    size = max(a.shape[0], b.shape[0])
    aa = np.zeros((size, size), dtype=float)
    bb = np.zeros((size, size), dtype=float)
    aa[:a.shape[0], :a.shape[1]] = a
    bb[:b.shape[0], :b.shape[1]] = b
    diff = aa - bb
    return {
        "interaction_frobenius": float(np.linalg.norm(diff, ord="fro")),
        "interaction_l1": float(np.abs(diff).sum()),
    }


def community_profile(graph, community):
    community = set(community)
    subgraph = graph.subgraph(community).copy()
    degrees = np.array([degree for _, degree in subgraph.degree()], dtype=float)
    clustering_map = nx.clustering(graph, nodes=list(community)) if len(community) else {}
    triangles_map = nx.triangles(graph, nodes=list(community)) if len(community) else {}

    return {
        "size": len(community),
        "internal_edges": subgraph.number_of_edges(),
        "internal_density": safe_density(subgraph.number_of_nodes(), subgraph.number_of_edges()),
        "degree_values": degrees,
        "clustering_values": np.array(list(clustering_map.values()), dtype=float),
        "triangle_values": np.array(list(triangles_map.values()), dtype=float),
        "mean_degree": safe_mean(degrees),
        "mean_clustering": safe_mean(list(clustering_map.values())),
        "mean_triangles": safe_mean(list(triangles_map.values())),
    }


def profile_vector(profile, class_nodes):
    return np.array([
        profile["size"] / max(class_nodes, 1),
        0.0 if np.isnan(profile["internal_density"]) else profile["internal_density"],
        profile["mean_degree"] / max(class_nodes - 1, 1),
        0.0 if np.isnan(profile["mean_clustering"]) else profile["mean_clustering"],
        math.log1p(0.0 if np.isnan(profile["mean_triangles"]) else profile["mean_triangles"]),
    ], dtype=float)


def align_profiles(original_profiles, clone_profiles, original_class_nodes, clone_class_nodes):
    if len(original_profiles) == 0 or len(clone_profiles) == 0:
        return []

    original_vectors = np.vstack([profile_vector(profile, original_class_nodes) for profile in original_profiles])
    clone_vectors = np.vstack([profile_vector(profile, clone_class_nodes) for profile in clone_profiles])
    cost = np.linalg.norm(original_vectors[:, None, :] - clone_vectors[None, :, :], axis=2)
    row_ind, col_ind = linear_sum_assignment(cost)
    return list(zip(row_ind.tolist(), col_ind.tolist()))


def compare_class(original, clone, label):
    original_info = original["subcommunities"].get(label)
    clone_info = clone["subcommunities"].get(label)
    if original_info is None or clone_info is None:
        return None

    original_graph = original_info["graph"]
    clone_graph = clone_info["graph"]
    original_communities = original_info["communities"]
    clone_communities = clone_info["communities"]

    original_profiles = [community_profile(original_graph, community) for community in original_communities]
    clone_profiles = [community_profile(clone_graph, community) for community in clone_communities]
    pairs = align_profiles(
        original_profiles,
        clone_profiles,
        original_graph.number_of_nodes(),
        clone_graph.number_of_nodes(),
    )

    degree_distances = []
    clustering_distances = []
    triangle_distances = []
    size_errors = []

    for original_idx, clone_idx in pairs:
        original_profile = original_profiles[original_idx]
        clone_profile = clone_profiles[clone_idx]
        degree_distances.append(safe_wasserstein(original_profile["degree_values"], clone_profile["degree_values"]))
        clustering_distances.append(safe_wasserstein(original_profile["clustering_values"], clone_profile["clustering_values"]))
        triangle_distances.append(safe_wasserstein(original_profile["triangle_values"], clone_profile["triangle_values"]))
        size_errors.append(abs(original_profile["size"] - clone_profile["size"]))

    original_matrix = interaction_matrix(original_graph, original_communities)
    clone_matrix = interaction_matrix(clone_graph, clone_communities)
    matrix_distances = padded_matrix_distance(original_matrix, clone_matrix)

    original_modularity = safe_modularity(original_graph, original_communities)
    clone_modularity = safe_modularity(clone_graph, clone_communities)

    original_nodes = original_info["nodes"]
    clone_nodes = clone_info["nodes"]
    aligned_nodes = (
        original["num_nodes"] == clone["num_nodes"]
        and len(original_nodes) == len(clone_nodes)
        and set(original_nodes) == set(clone_nodes)
    )

    nmi = float("nan")
    ari = float("nan")
    if aligned_nodes:
        nodes = sorted(original_nodes)
        original_labels = original_info["partition"][nodes]
        clone_labels = clone_info["partition"][nodes]
        valid = (original_labels >= 0) & (clone_labels >= 0)
        if valid.sum() > 1:
            nmi = float(normalized_mutual_info_score(original_labels[valid], clone_labels[valid]))
            ari = float(adjusted_rand_score(original_labels[valid], clone_labels[valid]))

    row = {
        "model": clone["name"],
        "class": label,
        "original_class_nodes": original_graph.number_of_nodes(),
        "clone_class_nodes": clone_graph.number_of_nodes(),
        "original_subcommunities": len(original_communities),
        "clone_subcommunities": len(clone_communities),
        "matched_subcommunities": len(pairs),
        "original_modularity": original_modularity,
        "clone_modularity": clone_modularity,
        "modularity_abs_error": abs(original_modularity - clone_modularity) if not (np.isnan(original_modularity) or np.isnan(clone_modularity)) else float("nan"),
        "degree_wasserstein": safe_mean(degree_distances),
        "clustering_wasserstein": safe_mean(clustering_distances),
        "triangle_wasserstein": safe_mean(triangle_distances),
        "subcommunity_size_abs_error": safe_mean(size_errors),
        "nmi": nmi,
        "ari": ari,
        "aligned_nodes_for_nmi_ari": aligned_nodes,
    }
    row.update(matrix_distances)
    return row

## Comparar clones contra o original

Esta celula produz duas tabelas: uma por classe e outra agregada por modelo. A tabela agregada e a principal para discutir o resultado no artigo.

In [ ]:
metric_columns = [
    "modularity_abs_error",
    "degree_wasserstein",
    "clustering_wasserstein",
    "triangle_wasserstein",
    "interaction_frobenius",
    "interaction_l1",
    "subcommunity_size_abs_error",
    "nmi",
    "ari",
]

comparison_rows = []
for model_name, clone_record in clone_records.items():
    labels = sorted(set(original_record["subcommunities"].keys()) & set(clone_record["subcommunities"].keys()))
    for label in labels:
        row = compare_class(original_record, clone_record, label)
        if row is not None:
            comparison_rows.append(row)

comparison_by_class = pd.DataFrame(comparison_rows)
comparison_by_class.to_csv(TABLES_DIR / "comparison_by_class.csv", index=False)

aggregate_rows = []
for model_name, group in comparison_by_class.groupby("model"):
    weights = group["original_class_nodes"].to_numpy(dtype=float)
    aggregate = {"model": model_name, "classes": len(group), "weighted_by": "original_class_nodes"}

    for column in metric_columns:
        values = group[column].to_numpy(dtype=float)
        valid = np.isfinite(values)
        aggregate[column] = float(np.average(values[valid], weights=weights[valid])) if valid.any() else float("nan")

    aggregate["nmi_ari_valid_classes"] = int(group["aligned_nodes_for_nmi_ari"].sum())
    aggregate_rows.append(aggregate)

comparison_overall = pd.DataFrame(aggregate_rows).sort_values("interaction_frobenius")
comparison_overall.to_csv(TABLES_DIR / "comparison_overall.csv", index=False)

print(f"Tabelas salvas em: {TABLES_DIR}")
comparison_overall

## Gerar figuras

As figuras abaixo reproduzem a analise visual das metricas: barras normalizadas para as distancias principais, NMI/ARI quando aplicavel, e heatmap da distancia entre matrizes de interacao por classe.

In [ ]:
def save_normalized_distance_barplot(comparison_overall):
    distance_columns = [
        "modularity_abs_error",
        "degree_wasserstein",
        "clustering_wasserstein",
        "triangle_wasserstein",
        "interaction_frobenius",
    ]

    plot_df = comparison_overall.set_index("model")[distance_columns]
    normalized = plot_df.copy()
    for column in normalized.columns:
        max_value = normalized[column].max(skipna=True)
        if pd.notna(max_value) and max_value > 0:
            normalized[column] = normalized[column] / max_value

    ax = normalized.plot(kind="bar", figsize=(10, 4), width=0.82)
    ax.set_ylabel("Distancia normalizada (menor e melhor)")
    ax.set_xlabel("")
    ax.set_title("Preservacao quantitativa de subcomunidades")
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=False)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "subcommunity_distances_normalized.svg", bbox_inches="tight")
    plt.show()


def save_nmi_ari_barplot(comparison_overall):
    valid = comparison_overall.dropna(subset=["nmi", "ari"])
    if len(valid) == 0:
        print("NMI/ARI nao foram plotados porque nenhum clone manteve IDs de nos comparaveis.")
        return

    ax = valid.set_index("model")[["nmi", "ari"]].plot(kind="bar", figsize=(6, 3.5), width=0.75)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Score")
    ax.set_xlabel("")
    ax.set_title("Similaridade de particoes detectadas")
    ax.legend(frameon=False)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "subcommunity_nmi_ari.svg", bbox_inches="tight")
    plt.show()


def save_interaction_heatmap(comparison_by_class):
    heatmap_df = comparison_by_class.pivot(index="model", columns="class", values="interaction_frobenius")
    plt.figure(figsize=(8, 3.5))

    if HAS_SEABORN:
        sns.heatmap(heatmap_df, annot=True, fmt=".3f", cmap="viridis_r")
    else:
        plt.imshow(heatmap_df.values, aspect="auto")
        plt.colorbar(label="Frobenius")
        plt.yticks(range(len(heatmap_df.index)), heatmap_df.index)
        plt.xticks(range(len(heatmap_df.columns)), heatmap_df.columns)

    plt.title("Distancia entre matrizes de interacao de subcomunidades")
    plt.xlabel("Classe")
    plt.ylabel("")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "interaction_matrix_distance_heatmap.svg", bbox_inches="tight")
    plt.show()


save_normalized_distance_barplot(comparison_overall)
save_nmi_ari_barplot(comparison_overall)
save_interaction_heatmap(comparison_by_class)

print(f"Figuras salvas em: {FIGURES_DIR}")

## Exportar tabela LaTeX

A ultima celula cria uma tabela compacta para incluir no texto ou adaptar para o paper.

In [ ]:
display_columns = [
    "model",
    "modularity_abs_error",
    "degree_wasserstein",
    "clustering_wasserstein",
    "triangle_wasserstein",
    "interaction_frobenius",
    "nmi",
    "ari",
]

latex_path = TABLES_DIR / "comparison_overall_latex.tex"
comparison_overall[display_columns].to_latex(latex_path, index=False, float_format="%.4f")

compact_table = comparison_overall[display_columns].copy()
for column in compact_table.columns:
    if column != "model":
        compact_table[column] = compact_table[column].map(lambda value: "" if pd.isna(value) else f"{value:.4f}")

print(f"LaTeX salvo em: {latex_path}")
compact_table